In [1]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from spark_session import get_spark

spark = get_spark("FastFood-P3-DataFrames")
sc = spark.sparkContext

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("restaurant_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("city", StringType()),
    StructField("status", StringType()),
    StructField("amount_mad", DoubleType()),
    StructField("items_count", IntegerType()),
    StructField("distance_km", DoubleType()),
    StructField("courier_id", StringType()),
    StructField("event_time", TimestampType()),
])
restaurants_schema = StructType([
    StructField("restaurant_id", StringType(), False),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("category", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
])

ORDERS = "s3a://lake/bronze/history/orders_history.jsonl"
RESTOS = "s3a://lake/bronze/referentiel/restaurants.json"

orders = (spark.read.schema(orders_schema)
          .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssX")
          .json(ORDERS))
restaurants = spark.read.schema(restaurants_schema).json(RESTOS)

orders.printSchema()
orders.show(3, truncate=False)
print(orders.count(), restaurants.count())

https://repos.spark-packages.org added as a remote repository with the name: repo-1


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-27fdc383-4c94-47df-9db5-31ad77e4c18e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 630ms :: artifacts dl 24ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|

root
 |-- order_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- items_count: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



+-----------+-------------+-----------+----------+-------+----------+-----------+-----------+----------+-------------------+
|order_id   |restaurant_id|customer_id|city      |status |amount_mad|items_count|distance_km|courier_id|event_time         |
+-----------+-------------+-----------+----------+-------+----------+-----------+-----------+----------+-------------------+
|ORD-0000001|R-020        |C-04463    |Marrakech |CREATED|58.5      |1          |2.4        |NULL      |2026-11-09 00:00:31|
|ORD-0000002|R-012        |C-11850    |Casablanca|CREATED|167.0     |3          |5.1        |NULL      |2026-11-09 00:00:34|
|ORD-0000003|R-031        |C-03921    |Marrakech |CREATED|88.0      |1          |1.2        |NULL      |2026-11-09 00:01:22|
+-----------+-------------+-----------+----------+-------+----------+-----------+-----------+----------+-------------------+
only showing top 3 rows



50000 200


In [2]:
clean = orders.dropDuplicates(["order_id", "status"])
delivered = clean.filter(F.col("status") == "DELIVERED")

ca_df = (delivered.groupBy("city")
         .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
              F.count("*").alias("nb"),
              F.round(F.avg("amount_mad"), 2).alias("panier_moyen"))
         .orderBy(F.desc("ca_mad")))
ca_df.show()

clean.createOrReplaceTempView("orders")
spark.sql("""
    SELECT city,
           ROUND(SUM(amount_mad), 2) AS ca_mad,
           COUNT(*)                  AS nb,
           ROUND(AVG(amount_mad), 2) AS panier_moyen
    FROM orders
    WHERE status = 'DELIVERED'
    GROUP BY city
    ORDER BY ca_mad DESC
""").show()

+----------+---------+----+------------+
|      city|   ca_mad|  nb|panier_moyen|
+----------+---------+----+------------+
|Casablanca|1001411.5|6772|      147.88|
|     Rabat| 542624.0|3533|      153.59|
| Marrakech| 407148.0|2805|      145.15|
|    Tanger| 338924.5|2244|      151.04|
+----------+---------+----+------------+



[Stage 15:>                                                         (0 + 1) / 1]

+----------+---------+----+------------+
|      city|   ca_mad|  nb|panier_moyen|
+----------+---------+----+------------+
|Casablanca|1001411.5|6772|      147.88|
|     Rabat| 542624.0|3533|      153.59|
| Marrakech| 407148.0|2805|      145.15|
|    Tanger| 338924.5|2244|      151.04|
+----------+---------+----+------------+



In [3]:
def ca_par_ville_df():
    return (spark.read.schema(orders_schema).json(ORDERS)
            .dropDuplicates(["order_id", "status"])
            .filter(F.col("status") == "DELIVERED")
            .groupBy("city").agg(F.sum("amount_mad"))
            .collect())

ca_par_ville_df()                         # chauffe
temps = []
for _ in range(3):
    t = time.time(); ca_par_ville_df(); temps.append(time.time() - t)
print("Temps DataFrame (s) :", [round(x, 2) for x in temps], "→ médiane", round(sorted(temps)[1], 2))

[Stage 37:======================================>                   (2 + 1) / 3]

Temps DataFrame (s) : [1.96, 1.83, 2.12] → médiane 1.96


In [8]:
joined = delivered.join(restaurants.select("restaurant_id", "category", "name"),
                        on="restaurant_id", how="left")

ca_cat_ville = (joined.groupBy("city", "category")
                .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
                     F.count("*").alias("nb"))
                .orderBy("city", F.desc("ca_mad")))
ca_cat_ville.show(50)

[Stage 115:=================================================>       (7 + 1) / 8]

+----------+----------+--------+----+
|      city|  category|  ca_mad|  nb|
+----------+----------+--------+----+
|Casablanca|  marocain|189139.5|1242|
|Casablanca|    burger|185784.0|1246|
|Casablanca|     pizza|155626.0| 955|
|Casablanca|     sushi|141643.0| 645|
|Casablanca|     tacos| 88036.5| 696|
|Casablanca|    poulet| 62879.5| 447|
|Casablanca| asiatique| 58141.5| 339|
|Casablanca|  sandwich| 51147.0| 583|
|Casablanca|   healthy| 40303.5| 251|
|Casablanca|patisserie| 26560.0| 353|
|Casablanca|      NULL|  2151.0|  15|
| Marrakech|    poulet| 66386.5| 486|
| Marrakech|     pizza| 65033.0| 394|
| Marrakech|     sushi| 54375.0| 229|
| Marrakech|     tacos| 54288.0| 476|
| Marrakech|  marocain| 46838.5| 307|
| Marrakech|    burger| 36253.5| 235|
| Marrakech|   healthy| 28899.0| 174|
| Marrakech|  sandwich| 23378.5| 259|
| Marrakech| asiatique| 20879.0| 113|
| Marrakech|patisserie|  9366.0| 121|
| Marrakech|      NULL|  1451.0|  11|
|     Rabat|    burger|110855.5| 686|
|     Rabat|

In [5]:
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def distance_band(d):
    if d is None:
        return None
    if d < 2:
        return "court"
    if d <= 5:
        return "moyen"
    return "long"

band_native = (F.when(F.col("distance_km") < 2, "court")
                .when(F.col("distance_km") <= 5, "moyen")
                .otherwise("long"))

def bench(col_expr, label, runs=3):
    # groupBy force l'évaluation de la colonne (un simple count() la ferait élaguer par Catalyst)
    f = lambda: clean.withColumn("distance_band", col_expr).groupBy("distance_band").count().collect()
    f()
    t = []
    for _ in range(runs):
        s = time.time(); f(); t.append(time.time() - s)
    print(f"{label:<18} médiane {sorted(t)[1]:.2f} s")

clean.cache().count()
bench(distance_band(F.col("distance_km")), "UDF Python")
bench(band_native, "when/otherwise")

clean.withColumn("distance_band", band_native).groupBy("distance_band").count().show()
clean.withColumn("distance_band", distance_band("distance_km")).select("distance_band").explain()
clean.withColumn("distance_band", band_native).select("distance_band").explain()

UDF Python         médiane 0.86 s
when/otherwise     médiane 0.64 s
+-------------+-----+
|distance_band|count|
+-------------+-----+
|         long| 5729|
|        court|12498|
|        moyen|30735|
+-------------+-----+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [pythonUDF0#3777 AS distance_band#3664]
   +- BatchEvalPython [distance_band(distance_km#7)#3663], [pythonUDF0#3777]
      +- InMemoryTableScan [distance_km#7]
            +- InMemoryRelation [order_id#0, restaurant_id#1, customer_id#2, city#3, status#4, amount_mad#5, items_count#6, distance_km#7, courier_id#8, event_time#9], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- AdaptiveSparkPlan isFinalPlan=false
                     +- SortAggregate(key=[order_id#0, status#4], functions=[first(restaurant_id#1, false), first(customer_id#2, false), first(city#3, false), first(amount_mad#5, false), first(items_count#6, false), first(distance_km#7, false), first(courier_id#8, false),

### UDF Python ou expression native
| Version | Temps médian |
|---|---|
| UDF `distance_band` | 0,86 s |
| `when/otherwise` | 0,64 s |

Résultat identique pour les deux versions : court 12 498, moyen 30 735, long 5 729.

Le plan physique de la version UDF contient un nœud **`BatchEvalPython`**. Pour chaque lot de lignes, Spark doit :
1. sérialiser les valeurs depuis la JVM (format Tungsten) vers un processus Python, via des sockets et Pickle ou Arrow ;
2. exécuter la fonction Python ligne par ligne ;
3. renvoyer et désérialiser les résultats vers la JVM.

L'UDF est en plus une **boîte noire pour Catalyst** : il ne peut ni l'analyser, ni la simplifier, ni la combiner avec d'autres expressions, ni la pousser vers la source.

La version `when/otherwise` est traduite en une expression **`CASE WHEN`** native, dans un simple `Project`. Catalyst la connaît et peut l'optimiser, et Tungsten la compile en bytecode JVM par *whole-stage code generation*. Elle s'exécute dans l'executor, sans aller-retour vers Python ni sérialisation, d'où un gain d'environ 25 %. Règle à retenir : n'utiliser une UDF Python qu'en dernier recours, quand aucune fonction de `pyspark.sql.functions` ne fait le travail.

On note aussi dans les deux plans l'`InMemoryRelation`, qui correspond au `cache()` de `clean`, et l'`Exchange hashpartitioning(order_id, status, 8)` : c'est le shuffle du `dropDuplicates`.

In [6]:
silver = (clean
          .join(restaurants.select("restaurant_id", "name", "category"), "restaurant_id", "left")
          .withColumn("distance_band", band_native)
          .withColumn("date", F.to_date("event_time")))

(silver.repartition("city").write.mode("overwrite")
       .partitionBy("city")
       .parquet("s3a://lake/silver/orders_history/"))

spark.read.parquet("s3a://lake/silver/orders_history/").groupBy("city").count().show()

[Stage 110:=========================================================(4 + 0) / 4]

+----------+-----+
|      city|count|
+----------+-----+
|Casablanca|21629|
| Marrakech| 8840|
|     Rabat|11244|
|    Tanger| 7249|
+----------+-----+



In [7]:
joined.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [restaurant_id])
:- Filter (status#4 = DELIVERED)
:  +- Deduplicate [order_id#0, status#4]
:     +- Relation [order_id#0,restaurant_id#1,customer_id#2,city#3,status#4,amount_mad#5,items_count#6,distance_km#7,courier_id#8,event_time#9] json
+- Project [restaurant_id#20, category#23, name#21]
   +- Relation [restaurant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
restaurant_id: string, order_id: string, customer_id: string, city: string, status: string, amount_mad: double, items_count: int, distance_km: double, courier_id: string, event_time: timestamp, category: string, name: string
Project [restaurant_id#1, order_id#0, customer_id#2, city#3, status#4, amount_mad#5, items_count#6, distance_km#7, courier_id#8, event_time#9, category#23, name#21]
+- Join LeftOuter, (restaurant_id#1 = restaurant_id#20)
   :- Filter (status#4 = DELIVERED)
   :  +- Deduplicate [order_id#0, status#4]
   :    

### Lecture du plan de la jointure

**1. Parsed Logical Plan** : la traduction brute du code, pas encore vérifiée. `'Join UsingJoin(LeftOuter, [restaurant_id])` porte une apostrophe, qui signale que les colonnes ne sont pas encore résolues.

**2. Analyzed Logical Plan** : l'Analyzer résout les noms et les types grâce au catalogue et aux schémas explicites. Chaque colonne reçoit un identifiant unique (`restaurant_id#1` côté commandes, `restaurant_id#20` côté référentiel). La clause `USING` devient la condition `restaurant_id#1 = restaurant_id#20`, et le schéma de sortie est entièrement typé.

**3. Optimized Logical Plan** : les règles de Catalyst s'appliquent.
- **Inférence de prédicats** : ajout de `isnotnull(status)` et `isnotnull(restaurant_id)`. Une valeur nulle ne peut pas satisfaire une égalité, donc on l'élimine au plus tôt.
- **Élagage de colonnes** : côté référentiel, seules `restaurant_id`, `name` et `category` sont gardées. `city`, `latitude` et `longitude` ne seront jamais lues.
- **Réutilisation du cache** : la branche commandes est remplacée par l'`InMemoryRelation` de `clean`, donc pas de relecture du JSON.

**4. Physical Plan** : Catalyst choisit la stratégie d'exécution.
- **`BroadcastHashJoin … LeftOuter, BuildRight`** : le référentiel (200 lignes, environ 29 Ko) est sous le seuil `spark.sql.autoBroadcastJoinThreshold` (10 Mo). Il est collecté puis **diffusé à chaque executor** (`BroadcastExchange`) sous forme de table de hachage. Chaque partition des commandes fait la jointure localement, **sans shuffle de la grande table**. Un `SortMergeJoin` aurait demandé de redistribuer et trier les deux côtés sur `restaurant_id`.
- **Pushdown vers la source** : le `FileScan json` du référentiel affiche `ReadSchema: struct<restaurant_id, name, category>` et `PushedFilters: [IsNotNull(restaurant_id)]`. La projection et le filtre sont appliqués dès la lecture.
- Le filtre `status = DELIVERED` est appliqué sur le cache (`InMemoryTableScan … [(status = DELIVERED)]`).
- Le seul `Exchange hashpartitioning(order_id, status, 200)` restant est celui du `dropDuplicates`, pas celui de la jointure.
- `AdaptiveSparkPlan isFinalPlan=false` : l'**AQE** pourra encore ajuster le plan à l'exécution, par exemple en fusionnant les petites partitions de shuffle.

In [9]:
spark.stop()

### Plan exécuté de la requête « CA par ville » (Spark UI, Query 9, job 22)
- **Scan json : 15 672 lignes en sortie** (sur 50 000). Catalyst a **poussé le filtre `status = 'DELIVERED'` sous le dédoublonnage** : le filtre porte sur une colonne de la clé de `dropDuplicates`, donc filtrer avant ou après donne le même résultat. On dédoublonne ainsi 3 fois moins de lignes.
- **SortAggregate + Exchange (15 354 lignes)** : c'est le `dropDuplicates(order_id, status)`, avec 15 672 → 15 354, soit 318 doublons DELIVERED retirés. L'Exchange est le shuffle sur la clé de dédoublonnage (environ 248 KiB).
- **HashAggregate (partial) → 4 lignes → Exchange → HashAggregate (final)** : c'est l'agrégation en deux phases, l'équivalent du combiner de `reduceByKey`. Seulement **4 enregistrements (252 octets)** traversent le second shuffle.
- **WholeStageCodegen (1) à (4)** : Tungsten fusionne les opérateurs de chaque bloc en une seule fonction Java compilée.
- **AQEShuffleRead, « number of coalesced partitions: 1 »** : l'AQE a mesuré la taille réelle des données (environ 260 KiB) et fusionné les 8 partitions de shuffle en 1, ce qui évite de lancer des tâches presque vides. Le job 22 n'exécute que son dernier stage : les 2 premiers sont « skipped », car leurs sorties de shuffle existent déjà.

### Plan exécuté de la jointure (Spark UI, Query 23)
- **Branche gauche** (commandes) : `InMemoryTableScan` (48 962 lignes, le DataFrame dédoublonné mis en cache) → Filter `status = DELIVERED` → **15 354 lignes**.
- **Branche droite** (référentiel) : Scan json (200 lignes, 29,2 KiB) → Filter `isnotnull` → **BroadcastExchange** : le driver collecte les 200 lignes, construit une table de hachage (environ 1 Mo) et l'envoie à chaque executor.
- **BroadcastHashJoin → 15 354 lignes** : chaque partition des commandes fait la jointure **localement**, dans le même bloc WholeStageCodegen. La grande table **n'est pas shufflée** pour la jointure. La jointure est **gauche** (`LeftOuter`) : les commandes vers des restaurants absents du référentiel (R-9xx) sont conservées, avec une catégorie `NULL`.
- **HashAggregate partial → 342 lignes** → Exchange de **14,8 KiB** seulement → AQEShuffleRead (1 partition) → HashAggregate final → **44 lignes** (4 villes × 10 catégories, plus 4 lignes `NULL`) → `TakeOrderedAndProject` (le `orderBy` + `show`).